# **Energybae_Solar_Load_Calculator**

# Install Required Libraries

In [1]:
!pip install google-generativeai openpyxl pillow -q

# Upload Electricity Bill

In [31]:
from google.colab import files

print("Upload your Electricity Bill:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
file_bytes = uploaded[filename]

ext = filename.split(".")[-1].lower()
mime_map = {"pdf":"application/pdf","jpg":"image/jpeg",
            "jpeg":"image/jpeg","png":"image/png"}
media_type = mime_map.get(ext, "image/jpeg")

print(f"File uploaded: {filename}")

Upload your Electricity Bill:


Saving Copy of WhatsApp Image 2026-02-12 at 13.48.47 (1).jpeg to Copy of WhatsApp Image 2026-02-12 at 13.48.47 (1) (1).jpeg
File uploaded: Copy of WhatsApp Image 2026-02-12 at 13.48.47 (1) (1).jpeg


# Set API Key

In [30]:
API_KEY = "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"  # Yahan apni Gemini key paste karo
print(" API Key set!")

 API Key set!


# Check Available Free Models

In [29]:
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

models = client.models.list()
free_models = [m.id for m in models.data if "free" in m.id]

print(" Free models available:")
for m in free_models:
    print(f"  {m}")

 Free models available:
  inclusionai/ring-2.6-1t:free
  baidu/cobuddy:free
  nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
  poolside/laguna-xs.2:free
  poolside/laguna-m.1:free
  baidu/qianfan-ocr-fast:free
  google/gemma-4-26b-a4b-it:free
  google/gemma-4-31b-it:free
  arcee-ai/trinity-large-thinking:free
  nvidia/nemotron-3-super-120b-a12b:free
  minimax/minimax-m2.5:free
  openrouter/free
  liquid/lfm-2.5-1.2b-thinking:free
  liquid/lfm-2.5-1.2b-instruct:free
  nvidia/nemotron-3-nano-30b-a3b:free
  nvidia/nemotron-nano-12b-v2-vl:free
  qwen/qwen3-next-80b-a3b-instruct:free
  nvidia/nemotron-nano-9b-v2:free
  openai/gpt-oss-120b:free
  openai/gpt-oss-20b:free
  z-ai/glm-4.5-air:free
  qwen/qwen3-coder:free
  cognitivecomputations/dolphin-mistral-24b-venice-edition:free
  meta-llama/llama-3.3-70b-instruct:free
  meta-llama/llama-3.2-3b-instruct:free
  nousresearch/hermes-3-llama-3.1-405b:free


# AI Bill Extraction Function

In [28]:
import base64, json, io
from openai import OpenAI
import PIL.Image

def extract_bill_data(file_bytes, media_type, api_key):
    client = OpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1"
    )

    prompt = """You are an expert at reading MSEDCL electricity bills.
Extract all fields and return ONLY valid JSON, no markdown, no explanation.
Keep all string values SHORT (under 50 characters).
Return exactly this structure:
{
  "consumer_name": "...",
  "consumer_number": "...",
  "billing_unit": "...",
  "tariff_rate": "...",
  "meter_number": "...",
  "reading_group": "...",
  "sanctioned_load_kw": 0,
  "security_deposit": 0,
  "bill_month": "...",
  "bill_date": "...",
  "due_date": "...",
  "current_reading": 0,
  "previous_reading": 0,
  "units_consumed": 0,
  "total_bill_amount": 0,
  "monthly_units": {
    "Feb-2025": 0,
    "Mar-2025": 0,
    "Apr-2025": 0,
    "May-2025": 0,
    "Jun-2025": 0,
    "Jul-2025": 0,
    "Aug-2025": 0,
    "Sep-2025": 0,
    "Oct-2025": 0,
    "Nov-2025": 0,
    "Dec-2025": 0,
    "Jan-2026": 0
  }
}"""

    b64 = base64.standard_b64encode(file_bytes).decode()
    image_url = f"data:{media_type};base64,{b64}"

    response = client.chat.completions.create(
        model="nvidia/nemotron-nano-12b-v2-vl:free",
        max_tokens=2000,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": image_url}}
            ]
        }]
    )

    text = response.choices[0].message.content.strip()

    # Raw response dekho
    print("Raw response:")
    print(text[:500])

    # Clean karo
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    text = text.strip()

    # JSON fix karo agar incomplete ho
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Incomplete JSON ko complete karo
        if not text.endswith("}"):
            text = text + '}}'
        try:
            return json.loads(text)
        except:
            # Manual fallback
            print("JSON incomplete - Fixing it Manually...")
            import re
            numbers = re.findall(r'"(\w[^"]+)":\s*(\d+\.?\d*)', text)
            strings = re.findall(r'"(\w[^"]+)":\s*"([^"]*)"', text)
            result = {}
            for k, v in strings:
                result[k] = v
            for k, v in numbers:
                result[k] = float(v) if '.' in v else int(v)
            if "monthly_units" not in result:
                result["monthly_units"] = {}
            return result

print(" Function ready!")

 Function ready!


# Extract Data from Bill using AI

In [27]:
print("AI is reading Bill... Plz just wait...")

data = extract_bill_data(file_bytes, media_type, API_KEY)

print("\n Data Extract completed!")
print("=" * 40)
print(f"Consumer Name    : {data.get('consumer_name')}")
print(f"Consumer Number  : {data.get('consumer_number')}")
print(f"Bill Month       : {data.get('bill_month')}")
print(f"Units Consumed   : {data.get('units_consumed')}")
print(f"Total Bill (Rs.) : {data.get('total_bill_amount')}")
print(f"Sanctioned Load  : {data.get('sanctioned_load_kw')} kW")
print(f"Bill Date        : {data.get('bill_date')}")
print(f"Due Date         : {data.get('due_date')}")
print("=" * 40)
print("\n Monthly Units:")
for month, units in data.get("monthly_units", {}).items():
    print(f"  {month} : {units}")

AI is reading Bill... Plz just wait...
Raw response:
{
  "consumer_name": "RANJANA MADHUSHAM KHOBRA",
  "consumer_number": "65000048255560",
  "billing_unit": "1.00",
  "tariff_rate": "0",
  "meter_number": "184244",
  "reading_group": "E2",
  "sanctioned_load_kw": 0,
  "security_deposit": 0,
  "bill_month": "Jan-2026",
  "bill_date": "01-Jan-2026",
  "due_date": "2026-02-01",
  "current_reading": 0,
  "previous_reading": 0,
  "units_consumed": 0,
  "total_bill_amount": 3450.00,
  "monthly_units": {
    "Feb-2025": 0,
    "Mar-2025": 0,
    "Apr-2

 Data Extract completed!
Consumer Name    : RANJANA MADHUSHAM KHOBRA
Consumer Number  : 65000048255560
Bill Month       : Jan-2026
Units Consumed   : 0
Total Bill (Rs.) : 3450.0
Sanctioned Load  : 0 kW
Bill Date        : 01-Jan-2026
Due Date         : 2026-02-01

 Monthly Units:
  Feb-2025 : 0
  Mar-2025 : 0
  Apr-2025 : 0
  May-2025 : 0
  Jun-2025 : 0
  Jul-2025 : 0
  Aug-2025 : 0
  Sep-2025 : 0
  Oct-2025 : 0
  Nov-2025 : 0
  Dec-2025 : 0


# Solar Excel Report Generator Function

In [25]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import io

def create_solar_excel(data):
    wb = Workbook()
    ws = wb.active
    ws.title = "Solar Load Analysis"

    def cs(ref, val, bold=False, bg=None, fc="000000", align="left", size=10):
        c = ws[ref]
        c.value = val
        c.font = Font(bold=bold, color=fc, size=size, name="Arial")
        c.alignment = Alignment(horizontal=align, vertical="center", wrap_text=True)
        if bg:
            c.fill = PatternFill("solid", start_color=bg)
        t = Side(style="thin", color="BDBDBD")
        c.border = Border(left=t, right=t, top=t, bottom=t)

    def mt(rng, val, bg="1B5E20", fc="FFFFFF", size=12):
        ws.merge_cells(rng)
        c = ws[rng.split(":")[0]]
        c.value = val
        c.font = Font(bold=True, color=fc, size=size, name="Arial")
        c.alignment = Alignment(horizontal="center", vertical="center")
        c.fill = PatternFill("solid", start_color=bg)

    for col, w in {"A":30,"B":22,"C":30,"D":22,"E":15,"F":15}.items():
        ws.column_dimensions[col].width = w

    ws.row_dimensions[1].height = 38
    mt("A1:F1", "ENERGYBAE — SOLAR LOAD CALCULATOR REPORT", size=14)
    mt("A2:F2", f"Consumer: {data.get('consumer_name','N/A')}  |  Bill: {data.get('bill_month','N/A')}", bg="2E7D32", size=10)

    row = 4
    mt(f"A{row}:F{row}", "SECTION 1 — CONSUMER DETAILS", bg="388E3C", size=11)
    row += 1

    for l1,v1,l2,v2 in [
        ("Consumer Name",       data.get("consumer_name","N/A"),    "Consumer Number",    data.get("consumer_number","N/A")),
        ("Billing Unit",        data.get("billing_unit","N/A"),     "Tariff Code",        data.get("tariff_rate","N/A")),
        ("Meter Number",        data.get("meter_number","N/A"),     "Reading Group",      data.get("reading_group","N/A")),
        ("Sanctioned Load(kW)", data.get("sanctioned_load_kw","N/A"),"Security Deposit",  data.get("security_deposit","N/A")),
        ("Bill Date",           data.get("bill_date","N/A"),        "Due Date",           data.get("due_date","N/A")),
        ("Bill Month",          data.get("bill_month","N/A"),       "Total Bill (Rs.)",   data.get("total_bill_amount","N/A")),
    ]:
        ws.row_dimensions[row].height = 22
        cs(f"A{row}",l1,bold=True,bg="C8E6C9")
        cs(f"B{row}",v1,bg="FFFFFF",align="center")
        cs(f"C{row}",l2,bold=True,bg="C8E6C9")
        cs(f"D{row}",v2,bg="FFFFFF",align="center")
        ws.merge_cells(f"E{row}:F{row}")
        row += 1

    row += 1
    mt(f"A{row}:F{row}", "SECTION 2 — METER READING", bg="388E3C", size=11)
    row += 1
    for i,(lbl,val) in enumerate([
        ("Current Reading",  data.get("current_reading","N/A")),
        ("Previous Reading", data.get("previous_reading","N/A")),
        ("Units Consumed",   data.get("units_consumed","N/A")),
    ]):
        c1=get_column_letter(i*2+1); c2=get_column_letter(i*2+2)
        cs(f"{c1}{row}",lbl,bold=True,bg="C8E6C9",align="center")
        cs(f"{c2}{row}",val,bg="FFF9C4",align="center",bold=True,size=12)
    row += 1

    row += 1
    mt(f"A{row}:F{row}", "SECTION 3 — 12-MONTH HISTORY", bg="388E3C", size=11)
    row += 1
    for i,h in enumerate(["Month","Units","Month","Units","Month","Units"],start=1):
        cs(f"{get_column_letter(i)}{row}",h,bold=True,bg="4CAF50",fc="FFFFFF",align="center")
    row += 1

    months = ["Feb-2025","Mar-2025","Apr-2025","May-2025","Jun-2025","Jul-2025",
              "Aug-2025","Sep-2025","Oct-2025","Nov-2025","Dec-2025","Jan-2026"]
    monthly = data.get("monthly_units",{})

    for i in range(4):
        ws.row_dimensions[row].height = 20
        for j in range(3):
            idx=i*3+j; m=months[idx]; v=monthly.get(m,"N/A")
            cs(f"{get_column_letter(j*2+1)}{row}",m,bg="E3F2FD",align="center")
            cs(f"{get_column_letter(j*2+2)}{row}",v,bg="FFFFFF",align="center",bold=(idx==11))
        row += 1

    row += 1
    mt(f"A{row}:F{row}", "SECTION 4 — SOLAR RECOMMENDATION", bg="E65100", fc="FFFFFF", size=11)
    row += 1

    vals = [monthly.get(m) for m in months if monthly.get(m) not in (None,0)]
    avg       = sum(vals)/len(vals) if vals else 0
    avg_daily = avg/30
    solar_kw  = round(avg_daily/4.5*1.25, 2)
    panels    = max(1, round((solar_kw*1000)/450))
    actual_kw = round(panels*450/1000, 2)
    savings   = round(avg*12*7.5)
    payback   = round(actual_kw*60000/savings,1) if savings else "N/A"
    co2       = round(avg*12*0.82)

    for lbl,val,bg in [
        ("Avg Monthly Units (kWh)",         round(avg,1),       "FFF3E0"),
        ("Avg Daily Consumption (kWh/day)", round(avg_daily,2),  "FFF9C4"),
        ("Recommended Solar Capacity (kW)", solar_kw,            "C8E6C9"),
        ("Panel Wattage Assumed",           "450W Mono PERC",    "E3F2FD"),
        ("No. of Panels Required",          panels,              "FFF3E0"),
        ("Total Installed Capacity (kW)",   actual_kw,           "C8E6C9"),
        ("System Type",                     "On-Grid",           "E3F2FD"),
        ("Est. Annual Savings (Rs.)",       f"Rs.{savings:,}",   "C8E6C9"),
        ("Payback Period",                  f"{payback} yrs",    "FFF9C4"),
        ("CO2 Offset/Year (kg)",            co2,                 "E3F2FD"),
    ]:
        ws.row_dimensions[row].height = 22
        ws.merge_cells(f"A{row}:C{row}")
        cs(f"A{row}",lbl,bold=True,bg=bg,size=10)
        ws.merge_cells(f"D{row}:F{row}")
        cs(f"D{row}",val,bold=True,bg="FFFFFF",align="center",size=11)
        row += 1

    row += 1
    mt(f"A{row}:F{row}", "www.energybae.in  |  energybae.co@gmail.com  |  +91 9112233120", bg="1B5E20", size=9)

    buf = io.BytesIO()
    wb.save(buf)
    buf.seek(0)
    return buf.getvalue()

print("Excel function ready!")

Excel function ready!


# Generate and Download Excel Report

In [24]:
from google.colab import files

print("Excel report are Creating...")
excel_bytes = create_solar_excel(data)

# File save karo
name  = str(data.get("consumer_name","Consumer")).replace(" ","_")
month = str(data.get("bill_month","")).replace(" ","_")
filename = f"Solar_Report_{name}_{month}.xlsx"

with open(filename, "wb") as f:
    f.write(excel_bytes)

print(f" Excel ready: {filename}")
print("Download started...")

files.download(filename)

Excel report are Creating...
 Excel ready: Solar_Report_RANJANA_MADHUS_HAM_KHOBRA_GADE_30-01-2026.xlsx
Download started...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Requirements.txt File

In [33]:
# Create Requirements.txt
content = """openai
openpyxl
pillow
google-generativeai
"""

with open("requirements.txt", "w") as f:
    f.write(content)

print("✅ requirements.txt ready!")

from google.colab import files
files.download("requirements.txt")

✅ requirements.txt ready!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Readme.md File

In [37]:
lines = [
    "# Energybae Solar Load Calculator\n",
    "\n",
    "## Overview\n",
    "AI-powered tool that reads MSEDCL electricity bills\n",
    "and generates Solar Load Calculator Excel report automatically.\n",
    "\n",
    "## What It Does\n",
    "1. User uploads MSEDCL electricity bill (JPG/PNG/PDF)\n",
    "2. AI reads and extracts all key data from the bill\n",
    "3. Solar system size is calculated automatically\n",
    "4. Formatted Excel report is downloaded instantly\n",
    "\n",
    "## Tools Used\n",
    "- Google Colab\n",
    "- OpenRouter API (Free)\n",
    "- NVIDIA Nemotron Vision Language Model\n",
    "- openpyxl (Excel generation)\n",
    "- Pillow (image processing)\n",
    "\n",
    "## How To Run\n",
    "1. Open Google Colab\n",
    "2. Run Cell 1 to install libraries\n",
    "3. Run Cell 2 to upload your bill\n",
    "4. Run Cell 3 to set your API key\n",
    "5. Run remaining cells one by one\n",
    "6. Excel report downloads automatically\n",
    "\n",
    "## Solar Calculation Logic\n",
    "- Avg Monthly Units = Sum of 12 months / 12\n",
    "- Solar Capacity (kW) = Avg daily units / 4.5 x 1.25\n",
    "- No. of Panels = Solar kW x 1000 / 450W\n",
    "- Annual Savings = Avg units x 12 x Rs 7.5\n",
    "- Payback Period = Installed cost / Annual savings\n",
    "- CO2 Offset = Avg units x 12 x 0.82 kg\n",
    "\n",
    "## Excel Report Structure\n",
    "- Section 1 — Consumer Details\n",
    "- Section 2 — Meter Reading\n",
    "- Section 3 — 12-Month Consumption History\n",
    "- Section 4 — Solar System Recommendation\n",
    "\n",
    "## What I Would Improve Next\n",
    "- Add Streamlit web interface\n",
    "- Batch processing for multiple bills\n",
    "- Better solar formulas with roof and shading data\n",
    "- Auto email Excel report to customer\n",
    "\n",
    "## About\n",
    "Built for Energybae — Empowering People with Renewable Energy\n",
    "www.energybae.in | energybae.co@gmail.com | +91 9112233120\n",
]

with open("README.md", "w") as f:
    f.writelines(lines)

print("README.md ready!")

from google.colab import files
files.download("README.md")

✅ README.md ready!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>